# 01 — Model A: Feature Engineering, Training, and Inspection

This notebook implements and inspects **Model A** for the **NYAYA** disaster relief payment protection system.

### Key Objectives:
1. **Load Data**: Ingest synthetic transaction logs (`nyaya_v2.csv.gz`).
2. **Feature Extraction**: Transform raw transactions into the Model A feature matrix via `data.features.tx_features_frame`.
3. **Seeded Partitioning**: Partition data using the exact seeded stratified split from `models.train_a` (Train 60%, Validation 10%, Isotonic Calibration 10%, Conformal Calibration 10%, Test 10%).
4. **Train Booster**: Train an XGBoost booster (`binary:logistic`, `aucpr` metric) using `models.train_a.PARAMS`.
5. **Evaluate Held-Out Test Set**: Measure PR-AUC, ROC-AUC, and Brier score across populations (normal, displaced victims, ATO, syndicates).
6. **Feature Importance**: Analyze feature importance (Gain, Weight, Cover) with a specific focus on disaster context features (`in_disaster_zone`, `severity_tier`).

*Note: All logic reuses project code from `data.features`, `models.common`, and `models.train_a` without duplicating business logic.*

In [ ]:
import sys
import os
from pathlib import Path

# Dataset / model tag
TAG = "v2"

# Add project root to sys.path
cwd = Path.cwd()
for candidate in [cwd, cwd.parent, Path("/home/ibm-user/shared/nyaya-kaustub")]:
    if (candidate / "models").is_dir() and str(candidate) not in sys.path:
        sys.path.insert(0, str(candidate))
        break

# Resolve data and artifacts directories with fallback
DATA_DIR = Path(os.environ.get("NYAYA_DATA", "/home/ibm-user/shared/data"))
if not DATA_DIR.exists():
    DATA_DIR = Path.cwd().parent / "data"

ARTIFACTS_DIR = Path(os.environ.get("NYAYA_ARTIFACTS", "/home/ibm-user/shared/nyaya/artifacts"))
if not ARTIFACTS_DIR.exists():
    ARTIFACTS_DIR = Path.cwd().parent / "artifacts"

print(f"Dataset / Model Tag:      {TAG}")
print(f"Project root on sys.path: {sys.path[0]}")
print(f"Data directory:           {DATA_DIR} (exists: {DATA_DIR.exists()})")
print(f"Artifacts directory:      {ARTIFACTS_DIR} (exists: {ARTIFACTS_DIR.exists()})")

In [ ]:
import json
import numpy as np
import pandas as pd
import xgboost as xgb
from sklearn.metrics import average_precision_score, roc_auc_score

# Check matplotlib availability in the image without pip installing
try:
    import matplotlib.pyplot as plt
    MATPLOTLIB_AVAILABLE = True
except ImportError:
    MATPLOTLIB_AVAILABLE = False
    print("Notice: matplotlib not present in environment. Outputting tabular summaries.")

# Reusable project imports
from data.features import MODEL_A_FEATURES, DISASTER_CONTEXT, tx_features_frame
from models.common import SEED, brier, reliability, split_indices
from models.train_a import FRACTIONS, PARAMS, VERSION

print(f"Model A Target Version: {VERSION}")
print(f"Seeded Split:           {FRACTIONS} (seed={SEED})")
print(f"Model A Features ({len(MODEL_A_FEATURES)}): {MODEL_A_FEATURES}")
print(f"Disaster Context Features: {DISASTER_CONTEXT}")

## 1. Load Data (`nyaya_v2.csv.gz`)
Ingest synthetic payment transactions generated by `data.generate`. Each row represents a single payment event ordered by epoch timestamp `ts`.

In [ ]:
data_file = DATA_DIR / f"nyaya_{TAG}.csv.gz"
print(f"Loading transaction records from {data_file}...")
df = pd.read_csv(data_file)
print(f"Total payments: {len(df):,}")
print(f"Columns ({len(df.columns)}): {list(df.columns)}")

print("\n--- Population Breakdown and Ground-Truth Fraud Rates ---")
pop_summary = df.groupby("population").agg(
    total_count=("tx_id", "count"),
    fraud_count=("is_fraud", "sum"),
    fraud_rate=("is_fraud", "mean"),
    mean_amount=("amount", "mean"),
    disaster_zone_pct=("in_disaster_zone", "mean")
).reset_index()
print(pop_summary.to_string(index=False))
print(f"\nOverall dataset fraud rate: {df['is_fraud'].mean():.4%}")

## 2. Feature Extraction with `tx_features_frame`
Transform the raw transaction log into the Model A feature matrix (`MODEL_A_FEATURES`) using `data.features.tx_features_frame`.
This encapsulates:
- Core payment attributes: `type_code`, `amount`, balances, `geo_shift_km`, `new_device`, `hour`, `is_night`.
- Disaster context: `in_disaster_zone`, `severity_tier`.
- Derived accounting anomalies: `orig_balance_err`, `dest_balance_err`, `amount_to_balance`, `orig_drained`.

In [ ]:
print(f"Extracting features for {len(df):,} transactions via tx_features_frame...")
X = tx_features_frame(df)
y = df["is_fraud"].to_numpy(dtype=int)

print(f"Feature matrix dimensions: {X.shape}")
assert list(X.columns) == MODEL_A_FEATURES, "Mismatch in extracted feature columns"

print("\nSample extracted features (first 5 records):")
print(X.head())

## 3. Seeded Stratified Train/Val/Calibration/Test Split
Split data using the seeded stratified split from `models.train_a`:
- `train` (60%): Model parameter optimization
- `val` (10%): Early stopping validation
- `iso` (10%): Isotonic regression probability calibration
- `conf` (10%): Conformal prediction quantile estimation
- `test` (10%): Held-out evaluation
Stratification over `population | is_fraud` guarantees that all populations (including disaster victims and syndicate attacks) are proportionally represented across all partitions.

In [ ]:
strata = df["population"].astype(str) + "|" + df["is_fraud"].astype(str)
sp = split_indices(strata, FRACTIONS, seed=SEED)

print("--- Split Partition Sizes and Distribution ---")
for split_name, idx in sp.items():
    print(f"  {split_name:6s}: {len(idx):,} rows ({len(idx)/len(df):.1%}) | Fraud count: {y[idx].sum():,} ({y[idx].mean():.3%})")

d = {k: (X.iloc[v], y[v]) for k, v in sp.items()}

## 4. Train Model A XGBoost Booster
Construct `xgb.DMatrix` objects and train with `models.train_a.PARAMS`, running up to 600 boosting rounds with early stopping on the validation split.

In [ ]:
dtrain = xgb.DMatrix(d["train"][0], d["train"][1], feature_names=MODEL_A_FEATURES)
dval = xgb.DMatrix(d["val"][0], d["val"][1], feature_names=MODEL_A_FEATURES)

print(f"Training XGBoost booster (Version: {VERSION})...")
print(f"Hyperparameters: {PARAMS}")

booster = xgb.train(
    PARAMS,
    dtrain,
    num_boost_round=600,
    evals=[(dtrain, "train"), (dval, "val")],
    early_stopping_rounds=30,
    verbose_eval=50
)
print(f"\nTraining complete. Best iteration: {booster.best_iteration}")

## 5. Held-Out Test Evaluation
Evaluate raw Model A prediction scores on the held-out test split:
- Precision-Recall AUC (`average_precision_score`)
- ROC-AUC (`roc_auc_score`)
- Brier score (`brier(y, p)`)
- Population-level breakdown: compare scores for legitimate disaster victims (`displaced`) vs attacks (`ato`, `syndicate`).

In [ ]:
Xt, yt = d["test"]
dtest = xgb.DMatrix(Xt, feature_names=MODEL_A_FEATURES)
p_raw = booster.predict(dtest, iteration_range=(0, booster.best_iteration + 1))

test_pr_auc = float(average_precision_score(yt, p_raw))
test_roc_auc = float(roc_auc_score(yt, p_raw))
test_brier = float(brier(yt, p_raw))

print("--- Held-Out Test Set Metrics (Raw Scores) ---")
print(f"PR-AUC:   {test_pr_auc:.4f}")
print(f"ROC-AUC:  {test_roc_auc:.4f}")
print(f"Brier:    {test_brier:.5f}")

pop_test = df["population"].iloc[sp["test"]].to_numpy()
pop_results = []
for p in sorted(set(pop_test)):
    m = pop_test == p
    pop_results.append({
        "population": p,
        "rows": int(m.sum()),
        "fraud_count": int(yt[m].sum()),
        "fraud_rate": float(yt[m].mean()),
        "mean_score": float(p_raw[m].mean()),
        "median_score": float(np.median(p_raw[m])),
        "p95_score": float(np.quantile(p_raw[m], 0.95))
    })

pop_df = pd.DataFrame(pop_results)
print("\n--- Performance and Score Distribution by Population ---")
print(pop_df.to_string(index=False))

## 6. Feature Importance Inspection
Analyze feature contribution across Gain, Weight (Frequency), and Cover.
In disaster relief settings, understanding whether predictions rely on transactional distress or disaster context is essential to prevent unfair freezes on vulnerable populations.

In [ ]:
gain_scores = booster.get_score(importance_type="gain")
weight_scores = booster.get_score(importance_type="weight")
cover_scores = booster.get_score(importance_type="cover")

importance_records = []
for f in MODEL_A_FEATURES:
    importance_records.append({
        "feature": f,
        "gain": float(gain_scores.get(f, 0.0)),
        "weight": int(weight_scores.get(f, 0)),
        "cover": float(cover_scores.get(f, 0.0)),
        "is_disaster_context": f in DISASTER_CONTEXT
    })

imp_df = pd.DataFrame(importance_records).sort_values("gain", ascending=False).reset_index(drop=True)
print("--- Model A Feature Importance (Sorted by Gain) ---")
print(imp_df.to_string(index=False))

if MATPLOTLIB_AVAILABLE:
    fig, ax = plt.subplots(figsize=(10, 6))
    colors = ["#e06666" if is_ctx else "#4a86e8" for is_ctx in imp_df["is_disaster_context"]]
    y_pos = np.arange(len(imp_df))
    ax.barh(y_pos, imp_df["gain"][::-1], color=colors[::-1], align="center")
    ax.set_yticks(y_pos)
    ax.set_yticklabels(imp_df["feature"][::-1])
    ax.set_xlabel("Average Gain")
    ax.set_title("Model A Feature Importance by Gain (Red = Disaster Context, Blue = Core Features)")
    plt.tight_layout()
    plt.show()

## 7. Artifact Consistency Verification
Verify output parity against `artifacts/metrics_a.json` when written by the pipeline.

In [ ]:
metrics_file = ARTIFACTS_DIR / "metrics_a.json"
meta_file = ARTIFACTS_DIR / "model_a_meta.json"

if meta_file.exists():
    meta = json.loads(meta_file.read_text())
    print(f"Loaded {meta_file.name}:")
    print(f"  Artifact version: {meta.get('version')}")
    print(f"  Features count:   {len(meta.get('features', []))}")
    print(f"  Legacy threshold: {meta.get('legacy_threshold')}")

if metrics_file.exists():
    saved_metrics = json.loads(metrics_file.read_text())
    print(f"\nLoaded {metrics_file.name}:")
    print(f"  Pipeline PR-AUC:           {saved_metrics['test'].get('pr_auc'):.4f}")
    print(f"  Pipeline ROC-AUC:          {saved_metrics['test'].get('roc_auc'):.4f}")
    print(f"  Pipeline Brier (Raw):      {saved_metrics['test'].get('brier_raw'):.5f}")
    print(f"  Pipeline Brier (Cal):      {saved_metrics['test'].get('brier_calibrated'):.5f}")
    print(f"  Conformal Coverage:        {saved_metrics['test']['conformal'].get('coverage'):.3%}")
else:
    print(f"\nNote: Artifacts in {ARTIFACTS_DIR} not yet present. Generated via `python -m models.train_a`.")